# MST embeddings across high-dimensional datasets

Compare 2D embeddings from the same MST settings on four image datasets with different feature counts: handwritten digits, MNIST, Fashion-MNIST, and Olivetti faces. The notebook standardizes a stratified sample from each dataset and displays each plot as its fit completes.

MNIST, Fashion-MNIST, and Olivetti may download on first use. Dataset loaders cache their downloads. Lower `MAX_SAMPLES` or `N_EPOCHS` for a faster run. When Apple MPS is available, the notebook uses it directly.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.datasets import fetch_openml, fetch_olivetti_faces, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "mst_embedding").is_dir()),
    None,
)
if repo_root is not None:
    sys.path.insert(0, str(repo_root))

from mst_embedding import IteratedMSTEmbedding


## Load and sample datasets

`MAX_SAMPLES` bounds the pairwise-distance workload. The full Olivetti dataset has 400 faces with 4,096 pixel features; the other datasets are sampled stratifiably up to the configured maximum.


In [ ]:
MAX_SAMPLES = 1000
RANDOM_STATE = 42


def stratified_standardized_sample(X, labels, max_samples=MAX_SAMPLES):
    X = np.asarray(X, dtype=np.float64)
    labels = np.asarray(labels)
    if len(X) > max_samples:
        indices, _ = train_test_split(
            np.arange(len(X)),
            train_size=max_samples,
            random_state=RANDOM_STATE,
            stratify=labels,
        )
        X = X[indices]
        labels = labels[indices]
    X = StandardScaler().fit_transform(X)
    return X, labels


def load_openml_dataset(name):
    dataset = fetch_openml(
        name,
        version=1,
        as_frame=False,
        parser="auto",
        cache=True,
    )
    return dataset.data, np.asarray(dataset.target, dtype=np.int64)


def load_digits_dataset():
    bunch = load_digits()
    return bunch.data, bunch.target


def load_olivetti_dataset():
    bunch = fetch_olivetti_faces(shuffle=True, random_state=RANDOM_STATE)
    return bunch.data, bunch.target


loaders = {
    "Digits (8×8)": load_digits_dataset,
    "MNIST (28×28)": lambda: load_openml_dataset("mnist_784"),
    "Fashion-MNIST (28×28)": lambda: load_openml_dataset("Fashion-MNIST"),
    "Olivetti faces (64×64)": load_olivetti_dataset,
}

datasets = {}
load_errors = {}
for name, loader in loaders.items():
    print(f"Loading {name} ...", flush=True)
    try:
        raw_X, raw_labels = loader()
        X, labels = stratified_standardized_sample(raw_X, raw_labels)
        datasets[name] = (X, labels)
        print(
            f"  using {len(X):,} samples × {X.shape[1]:,} features "
            f"across {len(np.unique(labels))} classes",
            flush=True,
        )
    except Exception as error:
        load_errors[name] = f"{type(error).__name__}: {error}"
        print(f"  skipped: {load_errors[name]}", flush=True)

if not datasets:
    raise RuntimeError("No datasets loaded successfully.")


## Fit and visualize

All datasets use the same estimator settings. Fits run sequentially and each scatter plot appears when its dataset is ready. Points are colored by class label; the labels are not used to train the embedding.


In [ ]:
N_MSTS = 8
N_EPOCHS = 1000
BATCH_SIZE = 4096
DEVICE = (
    "mps"
    if getattr(torch.backends, "mps", None) is not None
    and torch.backends.mps.is_available()
    else "cpu"
)

embeddings = {}
fit_summaries = []

for name, (X, labels) in datasets.items():
    print(f"Fitting {name} on {DEVICE} ...", flush=True)
    estimator = IteratedMSTEmbedding(
        n_msts=N_MSTS,
        n_components=2,
        n_epochs=N_EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=0.05,
        negative_ratio=4,
        lambda_rep=1.0,
        rank_weight_exponent=1.0,
        random_state=RANDOM_STATE,
        device=DEVICE,
    )
    started = time.perf_counter()
    embedding = estimator.fit_transform(X)
    elapsed = time.perf_counter() - started
    embeddings[name] = embedding
    fit_summaries.append({
        "dataset": name,
        "samples": len(X),
        "features": X.shape[1],
        "classes": len(np.unique(labels)),
        "n_msts": N_MSTS,
        "n_epochs": N_EPOCHS,
        "seconds": elapsed,
        "device": estimator.device_,
    })

    fig, ax = plt.subplots(figsize=(7, 5))
    points = ax.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=labels,
        cmap="tab20",
        s=12,
        alpha=0.8,
        linewidths=0,
    )
    ax.set_title(
        f"{name} — {len(X):,} samples × {X.shape[1]:,} features; "
        f"{N_MSTS} MSTs; {N_EPOCHS} epochs"
    )
    ax.set_xlabel("Embedding dimension 1")
    ax.set_ylabel("Embedding dimension 2")
    fig.colorbar(points, ax=ax, label="Class label")
    fig.tight_layout()
    display(fig)
    plt.close(fig)

fit_summary = pd.DataFrame(fit_summaries)
fit_summary
